# Logs for problem solving: find the cause of a checkout error spike

Companion notebook for [the article](https://www.elastic.co/observability-labs/blog/logs-for-problem-solving-checkout-errors).
Requires Elasticsearch and Kibana 9.5+ with a Platinum or Enterprise license (or a trial),
Docker, and a `.env` copied from `.env.example`.

Times are in UTC. The article shows them in UTC-5: 18:12 UTC is 13:12 in the article.

In [ ]:
%pip install -q "elasticsearch>=9.5,<10" python-dotenv==1.0.1 requests==2.32.3

## 1. Connect

In [ ]:
import os
import subprocess
import sys
import time
import warnings
from datetime import datetime

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv()

ES_URL = os.environ["ELASTICSEARCH_URL"]
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
SPACE = os.getenv("KIBANA_SPACE", "default")
KBN_BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"

# Names of everything this notebook creates. Clean up deletes exactly these.
STREAM = "logs-checkout_api.otel-default"
DATA_VIEW_ID = "logs-for-problem-solving-checkout-errors"  # the logs-* data view, only if you have none
LOG_FILE = "data/checkout-api.log"

EXPECTED_LINES = 137_306
INCIDENT = "2026-09-11T18:12:00Z"  # 13:12 in the article (UTC-5)

es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)
KBN_HEADERS = {
    "Authorization": f"ApiKey {API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}


def kbn(method, path, **kwargs):
    """Call a Kibana API in the configured space and raise on HTTP errors."""
    response = requests.request(
        method, f"{KBN_BASE}{path}", headers=KBN_HEADERS, timeout=120, **kwargs
    )
    response.raise_for_status()
    return response.json() if response.text else {}


def esql(query, title=None, show=True):
    """Run an ES|QL query, print the result as a table (unless show=False) and any warning, return the rows."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        response = es.esql.query(query=query)
    columns = [c["name"] for c in response["columns"]]
    rows = [dict(zip(columns, values)) for values in response["values"]]
    if show:
        if title:
            print(f"\n{title}")
        widths = [
            max(len(str(c)), *(len(str(r[c])) for r in rows)) if rows else len(c)
            for c in columns
        ]
        print("  ".join(str(c).ljust(w) for c, w in zip(columns, widths)))
        for r in rows:
            print("  ".join(str(r[c]).ljust(w) for c, w in zip(columns, widths)))
    for w in caught:
        # The helper queries below rely on the default LIMIT on purpose; print any other warning.
        if "No limit defined" not in str(w.message):
            print(f"warning: {w.message}")
    return rows


print(f"elasticsearch {es.info()['version']['number']} | space={SPACE}")

## 2. Generate the checkout-api logs and ship them with the EDOT Collector

In [ ]:
es.options(ignore_status=404).indices.delete_data_stream(name=STREAM)

subprocess.run([sys.executable, "generate_logs.py"], check=True)
subprocess.run(["docker", "compose", "up", "-d"], check=True)

count, deadline = 0, time.time() + 900
while time.time() < deadline:
    time.sleep(10)
    if es.indices.exists(index=STREAM):
        es.indices.refresh(index=STREAM)
        count = es.count(index=STREAM)["count"]
    print(f"indexed {count:,} / {EXPECTED_LINES:,}")
    if count >= EXPECTED_LINES:
        break
subprocess.run(["docker", "compose", "down"], check=True)
assert (
    count == EXPECTED_LINES
), f"expected {EXPECTED_LINES:,} documents, found {count:,}"

# Discover and log rate analysis in the article use a logs-* data view. Create it only if there is none.
existing = [
    v for v in kbn("GET", "/api/data_views")["data_view"] if v["title"] == "logs-*"
]
if existing:
    print(f"using existing logs-* data view {existing[0]['id']}")
else:
    try:
        kbn(
            "POST",
            "/api/data_views/data_view",
            json={
                "data_view": {
                    "id": DATA_VIEW_ID,
                    "title": "logs-*",
                    "name": "logs-*",
                    "timeFieldName": "@timestamp",
                }
            },
        )
        print(f"created logs-* data view {DATA_VIEW_ID}")
    except requests.HTTPError as error:
        # API keys without the Kibana privilege indexPatterns:manage cannot create data views.
        print(
            f"could not create the data view ({error.response.status_code}): create it in Kibana, "
            "Discover > Data view > Create a data view, index pattern logs-*, timestamp field @timestamp"
        )

In [ ]:
esql(
    f"""
FROM {STREAM}
| STATS lines = COUNT(*), errors = COUNT(*) WHERE severity_text == "ERROR",
        first_line = MIN(@timestamp), last_line = MAX(@timestamp), pods = COUNT_DISTINCT(k8s.pod.name)
""",
    "Lines in the three hours around the incident",
)

esql(
    f"""
FROM {STREAM}
| WHERE severity_text == "ERROR"
| STATS errors = COUNT(*) BY bucket = BUCKET(@timestamp, 5 minutes)
| EVAL period = CASE(bucket < TO_DATETIME("2026-09-11T18:10:00Z"), "before 18:10",
                     bucket >= TO_DATETIME("2026-09-11T18:15:00Z"), "after 18:15", "18:10-18:15")
| STATS avg_errors_per_5_min = ROUND(AVG(errors), 1), min = MIN(errors), max = MAX(errors) BY period
""",
    "Errors per five minutes (the histogram in Discover)",
)

_ = esql(
    f"""
FROM {STREAM}
| WHERE url.path == "/api/checkout/confirm" AND severity_text != "WARN"
| EVAL period = CASE(@timestamp < TO_DATETIME("{INCIDENT}"), "before 18:12", "after 18:12")
| STATS confirm_requests = COUNT(*), errors = COUNT(*) WHERE severity_text == "ERROR" BY period
| EVAL error_rate_pct = ROUND(100.0 * errors / confirm_requests, 2)
""",
    "Error rate on POST /api/checkout/confirm",
)

## 3. What is new: group the log lines into patterns

In [ ]:
patterns = esql(
    f"""
FROM {STREAM}
| STATS lines = COUNT(*) BY pattern = CATEGORIZE(message)
| SORT lines DESC
""",
    "Patterns over all lines",
)
print(f"\n{sum(p['lines'] for p in patterns):,} lines in {len(patterns)} patterns")

error_patterns = esql(
    f"""
FROM {STREAM}
| WHERE severity_text == "ERROR"
| STATS lines = COUNT(*), first_seen = MIN(@timestamp) BY pattern = CATEGORIZE(message)
| SORT lines DESC
""",
    "Patterns over the error lines",
)
print(
    f"\n{sum(p['lines'] for p in error_patterns):,} error lines in {len(error_patterns)} patterns"
)

## 4. When did it start: ES|QL CHANGE_POINT

In [ ]:
CHANGE_POINT_QUERY = f"""
FROM {STREAM}
| STATS errors = COUNT(*) WHERE severity_text == "ERROR" BY minute = BUCKET(@timestamp, 1 minute)
| SORT minute
| CHANGE_POINT errors ON minute
| LIMIT 1000
"""
series = esql(CHANGE_POINT_QUERY, show=False)

print(f"{len(series)} one-minute buckets")
for row in series:
    if row["type"]:
        print(
            f"change point: {row['type']} at {row['minute']} (p-value {row['pvalue']:.2g})"
        )
        incident_minute = row["minute"]

before = [r["errors"] for r in series if r["minute"] < incident_minute]
after = [r["errors"] for r in series if r["minute"] >= incident_minute]
print(
    f"errors per minute: {sum(before) / len(before):.1f} before, {sum(after) / len(after):.1f} after"
)

## 5. Where is it coming from: compare errors before and after the change point

In [ ]:
for field in ["k8s.pod.name", "service.version", "url.path", "error.type"]:
    esql(
        f"""
FROM {STREAM}
| WHERE severity_text == "ERROR"
| EVAL after_incident = @timestamp >= TO_DATETIME("{INCIDENT}")
| STATS before = COUNT(*) WHERE NOT after_incident, after = COUNT(*) WHERE after_incident BY {field}
| SORT after DESC
""",
        f"Errors by {field}",
    )

caps = es.field_caps(index=STREAM, fields="ecs.version")
print(f"\necs.version in {STREAM}: {'yes' if caps['fields'] else 'no'}")

## 6. Why: read the lines around the first timeout

In [ ]:
first = esql(
    f"""
FROM {STREAM}
| WHERE error.type == "PaymentGatewayTimeout"
| SORT @timestamp
| LIMIT 1
| KEEP @timestamp, k8s.pod.name, service.version, message
""",
    "First timeout",
)[0]

surrounding = esql(
    f"""
FROM {STREAM}
| WHERE k8s.pod.name == "{first['k8s.pod.name']}" AND @timestamp <= TO_DATETIME("{first['@timestamp']}")
| SORT @timestamp DESC
| KEEP @timestamp, severity_text, message
""",
    "Lines from the same pod up to the first timeout (newest first)",
)
print(f"\n{len(surrounding) - 1} older documents on {first['k8s.pod.name']}")

config = next(r for r in surrounding if r["message"].startswith("Loaded configuration"))
gap = datetime.fromisoformat(
    first["@timestamp"].replace("Z", "+00:00")
) - datetime.fromisoformat(config["@timestamp"].replace("Z", "+00:00"))
print(f"configuration loaded {gap.total_seconds():.1f} s before the first timeout")

_ = esql(
    f"""
FROM {STREAM}
| WHERE message LIKE "Payment gateway slow*" AND @timestamp < TO_DATETIME("{INCIDENT}")
| STATS warnings = COUNT(*), min_ms = MIN(payment.latency_ms), median_ms = MEDIAN(payment.latency_ms),
        max_ms = MAX(payment.latency_ms) BY k8s.pod.name, service.version
| SORT k8s.pod.name
""",
    "Slow payment warnings before 18:12 UTC",
)

## 7. Confirm before you act

In [ ]:
_ = esql(
    f"""
FROM {STREAM}
| WHERE severity_text == "ERROR"
| STATS errors = COUNT(*), first_seen = MIN(@timestamp) BY k8s.pod.name, service.version, error.type
| SORT errors DESC
| LIMIT 20
""",
    "Errors by pod, version and error type",
)

## 8. Clean up

In [ ]:
es.options(ignore_status=404).indices.delete_data_stream(name=STREAM)
response = requests.delete(
    f"{KBN_BASE}/api/data_views/data_view/{DATA_VIEW_ID}",
    headers=KBN_HEADERS,
    timeout=60,
)
print(
    f"data view {DATA_VIEW_ID}: {'deleted' if response.ok else 'not created by this notebook'}"
)
subprocess.run(["docker", "compose", "down", "-v"], check=True)
if os.path.exists(LOG_FILE):
    os.remove(LOG_FILE)
print("done")